In [ ]:
"""
=============================================================
  NSE PORTFOLIO BACKTEST — FCF Filter + Multi-Model Valuation
  ─────────────────────────────────────────────────────────
  FILTERS (stock must pass ALL):
    1. Free Cash Flow > 0  (FCF positive)
    2. FCF Yield > 2%      (not overvalued on FCF basis)
    3. EPS > 0             (profitable)
    4. Market cap in range

  VALUATION SCORE (composite of 3 models):
    1. Graham Intrinsic Value     → discount to price
    2. DCF (FCF-based)            → margin of safety
    3. EV/FCF ratio               → relative cheapness

  ENTRY SIGNAL:
    UTBot buy signal + stock below composite intrinsic value
    Ranked by: valuation_discount / volatility  (quality-adjusted)
=============================================================
"""

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import os
import pickle
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

# ─────────────────────────────────────────────
#  SETTINGS
# ─────────────────────────────────────────────
INITIAL_CAPITAL   = 100_000
STOP_LOSS_PCT     = 0.05    # 4% hard stop (from your new script)
MAX_POSITIONS     = 14    # 10 positions (from your new script)
LOOKBACK_PERIOD   = "10y"
ATR_PERIOD        = 4
ATR_MULTIPLIER    = 0.02
MAX_DRAWDOWN_PCT  = 0.03   # relaxed from 0.01 (too tight)
MIN_MARKET_CAP_CR = 1_00
MAX_MARKET_CAP_CR = 1_000_00

# FCF FILTER SETTINGS
MIN_FCF           = 0           # must be > 0 (FCF positive)
MIN_FCF_YIELD_PCT = 2.0         # FCF / Market Cap must be > 2%

# DCF SETTINGS
DCF_DISCOUNT_RATE = 0.12        # 12% WACC (typical for India)
DCF_TERMINAL_GROWTH = 0.05      # 5% terminal growth
DCF_YEARS         = 10          # projection years

# VALUATION WEIGHTS (must sum to 1.0)
W_GRAHAM          = 0.35        # Graham intrinsic weight
W_DCF             = 0.40        # DCF weight (most important)
W_EVFCF           = 0.25        # EV/FCF cheapness weight

MAX_WORKERS       = 20
CACHE_DIR         = "cache_fcf"
USE_CACHE         = True

# ─────────────────────────────────────────────
#  CACHE
# ─────────────────────────────────────────────
os.makedirs(CACHE_DIR, exist_ok=True)

def cache_path(ticker):
    return os.path.join(CACHE_DIR, ticker.replace(".", "_") + ".pkl")

def load_from_cache(ticker):
    p = cache_path(ticker)
    if USE_CACHE and os.path.exists(p):
        with open(p, "rb") as f:
            return pickle.load(f)
    return None

def save_to_cache(ticker, data):
    with open(cache_path(ticker), "wb") as f:
        pickle.dump(data, f)

# ─────────────────────────────────────────────
#  LOAD NSE TICKERS
# ─────────────────────────────────────────────
def load_nse_tickers():
    df = pd.read_csv("data/EQUITY_L.csv")
    symbols = df["SYMBOL"].dropna().astype(str).str.strip().unique().tolist()
    return [s + ".NS" for s in symbols if "&" not in s]

# ─────────────────────────────────────────────
#  UT BOT
# ─────────────────────────────────────────────
def compute_utbot(df):
    df = df.copy()
    df["tr"] = np.maximum(
        df["High"] - df["Low"],
        np.maximum(
            (df["High"] - df["Close"].shift()).abs(),
            (df["Low"]  - df["Close"].shift()).abs()
        )
    )
    df["atr"]  = df["tr"].ewm(span=ATR_PERIOD, adjust=False).mean()
    df["upper"] = df["Close"] - ATR_MULTIPLIER * df["atr"]
    df["lower"] = df["Close"] + ATR_MULTIPLIER * df["atr"]

    trend = [1]
    for i in range(1, len(df)):
        if   df["Close"].iloc[i] > df["lower"].iloc[i - 1]: trend.append(1)
        elif df["Close"].iloc[i] < df["upper"].iloc[i - 1]: trend.append(-1)
        else:                                                 trend.append(trend[-1])

    df["trend"]      = trend
    df["buy"]        = (df["trend"] == 1)  & (df["trend"].shift() == -1)
    df["sell"]       = (df["trend"] == -1) & (df["trend"].shift() == 1)
    df["volatility"] = df["Close"].pct_change(fill_method=None).rolling(20).std()
    return df

# ─────────────────────────────────────────────
#  FCF FILTER
#  Returns (passed: bool, fcf: float, fcf_yield: float)
# ─────────────────────────────────────────────
def check_fcf(info, cashflow_df):
    """
    FCF = Operating Cash Flow - Capital Expenditure
    Uses yfinance cashflow statement (annual).
    Falls back to info fields if statement unavailable.
    """
    fcf = None

    # Method 1: from cashflow statement (most accurate)
    if cashflow_df is not None and not cashflow_df.empty:
        try:
            # yfinance cashflow rows vary by ticker — try common row names
            ocf, capex = None, None
            for row in ["Operating Cash Flow", "Total Cash From Operating Activities",
                        "Cash Flow From Continuing Operating Activities"]:
                if row in cashflow_df.index:
                    ocf = cashflow_df.loc[row].iloc[0]  # most recent year
                    break
            for row in ["Capital Expenditure", "Purchase Of Property Plant And Equipment",
                        "Capital Expenditures"]:
                if row in cashflow_df.index:
                    capex = cashflow_df.loc[row].iloc[0]
                    break
            if ocf is not None and capex is not None:
                # capex is usually negative in statements → FCF = OCF + capex
                fcf = float(ocf) + float(capex)
        except Exception:
            pass

    # Method 2: fallback to info fields
    if fcf is None:
        ocf   = info.get("operatingCashflow")
        capex = info.get("capitalExpenditures")   # usually negative
        if ocf is not None and capex is not None:
            fcf = float(ocf) + float(capex)        # capex negative → adds

    # Method 3: freeCashflow directly from info
    if fcf is None:
        fcf_raw = info.get("freeCashflow")
        if fcf_raw is not None:
            fcf = float(fcf_raw)

    if fcf is None:
        return False, None, None

    # FCF must be positive
    if fcf <= MIN_FCF:
        return False, fcf, None

    # FCF Yield = FCF / Market Cap
    market_cap = info.get("marketCap")
    if not market_cap or market_cap <= 0:
        return False, fcf, None

    fcf_yield = (fcf / market_cap) * 100

    if fcf_yield < MIN_FCF_YIELD_PCT:
        return False, fcf, fcf_yield

    return True, fcf, fcf_yield

# ─────────────────────────────────────────────
#  VALUATION MODELS
# ─────────────────────────────────────────────
def graham_intrinsic(eps, growth_pct):
    """
    Benjamin Graham formula:
    V = EPS × (8.5 + 2g)
    where g = expected annual growth (capped at 12%)
    """
    g = min(growth_pct, 12.0)
    return eps * (8.5 + 2 * g)

def dcf_intrinsic(fcf, shares_outstanding, growth_pct):
    """
    Simple FCF-based DCF:
    Project FCF for DCF_YEARS at growth_pct,
    then add terminal value. Discount at DCF_DISCOUNT_RATE.
    Returns intrinsic value per share.
    """
    if shares_outstanding <= 0 or fcf <= 0:
        return None

    g  = min(growth_pct / 100, 0.20)   # cap at 20%
    r  = DCF_DISCOUNT_RATE
    gT = DCF_TERMINAL_GROWTH

    pv = 0.0
    cf = fcf
    for yr in range(1, DCF_YEARS + 1):
        cf  *= (1 + g)
        pv  += cf / (1 + r) ** yr

    # Terminal value (Gordon Growth)
    terminal_cf = cf * (1 + gT)
    terminal_pv = (terminal_cf / (r - gT)) / (1 + r) ** DCF_YEARS
    total_pv    = pv + terminal_pv

    return total_pv / shares_outstanding

def evfcf_score(ev, fcf):
    """
    EV/FCF ratio — lower is cheaper.
    Convert to a 'value' score: higher = cheaper.
    Score = 1 / (EV/FCF) capped sensibly.
    """
    if fcf <= 0 or ev is None or ev <= 0:
        return 0.0
    ratio = ev / fcf
    if ratio <= 0:
        return 0.0
    # Score: EV/FCF of 10 → score 0.10, EV/FCF of 5 → score 0.20
    return 1.0 / ratio

def composite_intrinsic(eps, fcf, shares, growth_pct, ev):
    """
    Weighted composite of Graham + DCF + EV/FCF score.
    Returns (composite_intrinsic_price, dcf_value, graham_value, evfcf_ratio)
    """
    graham = graham_intrinsic(eps, growth_pct)
    dcf    = dcf_intrinsic(fcf, shares, growth_pct)
    evfcf  = evfcf_score(ev, fcf)

    if dcf is None:
        # Fall back to Graham only
        return graham, None, graham, None

    evfcf_ratio = (ev / fcf) if fcf > 0 and ev else None

    # Normalise EV/FCF score to a price-like value for weighting
    # We scale DCF by EV/FCF cheapness: cheaper EV/FCF → boost DCF slightly
    evfcf_boost = 1.0 + (evfcf * 2)   # small boost, not dominant
    composite   = (W_GRAHAM * graham) + (W_DCF * dcf * evfcf_boost) + (W_EVFCF * dcf)

    return composite, dcf, graham, evfcf_ratio

# ─────────────────────────────────────────────
#  FETCH ONE TICKER
# ─────────────────────────────────────────────
def fetch_ticker(ticker):
    cached = load_from_cache(ticker)
    if cached is not None:
        return ticker, cached

    try:
        stock = yf.Ticker(ticker)
        info  = stock.info

        # ── Market cap filter ──
        market_cap = info.get("marketCap")
        if not market_cap:
            return ticker, None
        mc_cr = market_cap / 1e7
        if not (MIN_MARKET_CAP_CR <= mc_cr <= MAX_MARKET_CAP_CR):
            return ticker, None

        # ── EPS filter ──
        eps = info.get("trailingEps")
        if eps is None or eps <= 0:
            return ticker, None

        # ── FCF filter ──
        try:
            cf_df = stock.cashflow          # annual cashflow statement
        except Exception:
            cf_df = None

        fcf_ok, fcf, fcf_yield = check_fcf(info, cf_df)
        if not fcf_ok:
            return ticker, None

        # ── Price history ──
        df = stock.history(period=LOOKBACK_PERIOD)
        if df.empty or len(df) < 60:
            return ticker, None
        df = compute_utbot(df)

        # ── Valuation inputs ──
        growth_raw = info.get("earningsQuarterlyGrowth") or 0.05
        growth_pct = min(growth_raw * 100, 20.0)

        shares     = info.get("sharesOutstanding") or 1
        ev         = info.get("enterpriseValue")    # can be None

        intrinsic, dcf_val, graham_val, evfcf_ratio = composite_intrinsic(
            eps, fcf, shares, growth_pct, ev
        )

        # Attach all valuation columns
        df["Intrinsic"]    = max(intrinsic, 0)     # floor at 0
        df["DCF"]          = dcf_val if dcf_val else graham_val
        df["Graham"]       = graham_val
        df["EVFCFRatio"]   = evfcf_ratio if evfcf_ratio else np.nan
        df["FCF"]          = fcf
        df["FCFYield"]     = fcf_yield
        df["MarketCapCr"]  = mc_cr
        df["GrowthPct"]    = growth_pct

        save_to_cache(ticker, df)
        return ticker, df

    except Exception:
        return ticker, None

# ─────────────────────────────────────────────
#  PARALLEL DATA LOAD
# ─────────────────────────────────────────────
def load_all_data(tickers):
    all_data = {}
    total, done = len(tickers), 0
    print(f"\nFetching {total} tickers ({MAX_WORKERS} workers) with FCF filter …\n")
    t0 = time.time()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(fetch_ticker, t): t for t in tickers}
        for fut in as_completed(futures):
            ticker, df = fut.result()
            done += 1
            if df is not None:
                all_data[ticker] = df
            if done % 100 == 0 or done == total:
                elapsed = time.time() - t0
                eta     = (elapsed / done) * (total - done)
                print(f"  [{done}/{total}]  {done/total*100:.0f}%  "
                      f"|  passed FCF filter: {len(all_data)}  "
                      f"|  ETA: {eta/60:.1f} min")

    print(f"\nFCF-positive stocks loaded: {len(all_data)} / {total}  "
          f"({len(all_data)/total*100:.1f}%)  in {(time.time()-t0)/60:.1f} min\n")
    return all_data

# ─────────────────────────────────────────────
#  METRICS
# ─────────────────────────────────────────────
def calc_cagr(start, end, years):
    if years <= 0 or start <= 0: return 0.0
    return (end / start) ** (1 / years) - 1

def calc_max_drawdown(equity):
    eq = np.array(equity)
    return ((eq - np.maximum.accumulate(eq)) / np.maximum.accumulate(eq)).min()

def calc_sharpe(equity, rf=0.06):
    eq  = np.array(equity)
    ret = np.diff(eq) / eq[:-1]
    ex  = ret - rf / 252
    return (ex.mean() / ex.std()) * np.sqrt(252) if ex.std() > 0 else 0.0

# ─────────────────────────────────────────────
#  BACKTEST
# ─────────────────────────────────────────────
def run_backtest(all_data):
    cash           = float(INITIAL_CAPITAL)
    open_positions = {}
    trade_log      = []
    equity_curve   = []
    date_index     = []

    all_dates = sorted(set(d for df in all_data.values() for d in df.index))
    print(f"Running backtest over {len(all_dates)} trading days …\n")

    for current_date in all_dates:

        # ── SELL ──────────────────────────────
        for ticker in list(open_positions.keys()):
            df  = all_data[ticker]
            if current_date not in df.index:
                continue
            row = df.loc[current_date]
            pos = open_positions[ticker]

            pos["highest_price"] = max(pos["highest_price"], row["Close"])

            stop_price = pos["entry_price"] * (1 - STOP_LOSS_PCT)
            atr_stop   = pos["highest_price"] - (row["atr"] * ATR_MULTIPLIER)
            drawdown   = (pos["highest_price"] - row["Close"]) / pos["highest_price"]

            exit_reason = None
            if   row["Close"] <= stop_price:       exit_reason = "Stop Loss"
            elif row["Close"] <= atr_stop:         exit_reason = "ATR Trailing Stop"
            elif drawdown >= MAX_DRAWDOWN_PCT:      exit_reason = "Max Drawdown"
            elif row["sell"]:                       exit_reason = "UT Sell Signal"

            if exit_reason is None:
                continue

            exit_price = row["Close"]
            proceeds   = pos["shares"] * exit_price
            profit     = proceeds - pos["invested"]
            cash      += proceeds

            trade_log.append({
                "Stock":         ticker,
                "Entry Date":    pos["entry_date"],
                "Exit Date":     current_date,
                "Entry Price":   round(pos["entry_price"],  2),
                "Exit Price":    round(exit_price,           2),
                "Shares":        round(pos["shares"],        4),
                "Invested":      round(pos["invested"],      2),
                "Profit":        round(profit,               2),
                "Return %":      round((exit_price / pos["entry_price"] - 1) * 100, 2),
                "Exit Reason":   exit_reason,
                "Market Cap Cr": row["MarketCapCr"],
                "FCF Yield %":   round(row["FCFYield"], 2),
                "EV/FCF":        round(row["EVFCFRatio"], 1) if not pd.isna(row["EVFCFRatio"]) else None,
                "DCF Value":     round(row["DCF"],  2),
                "Graham Value":  round(row["Graham"], 2),
                "Intrinsic":     round(row["Intrinsic"], 2),
                "Holding Days":  (current_date - pos["entry_date"]).days,
            })
            del open_positions[ticker]

        # ── BUY ───────────────────────────────
        available_slots = MAX_POSITIONS - len(open_positions)

        if available_slots > 0 and cash > 0:
            candidates = []

            for ticker, df in all_data.items():
                if ticker in open_positions:
                    continue
                if current_date not in df.index:
                    continue
                row = df.loc[current_date]

                if pd.isna(row["volatility"]) or row["volatility"] == 0:
                    continue
                if not row["buy"]:
                    continue

                # Core entry condition: price below composite intrinsic
                if row["Close"] >= row["Intrinsic"]:
                    continue

                # Quality-adjusted score:
                # discount = how cheap vs intrinsic
                # fcf_yield boost = better FCF yield → higher score
                # divided by volatility = prefer less volatile discounts
                discount  = (row["Intrinsic"] - row["Close"]) / row["Intrinsic"]
                fcf_boost = row["FCFYield"] / 10.0      # normalize FCF yield
                score     = (discount + fcf_boost) / row["volatility"]

                candidates.append((ticker, score, row))

            candidates.sort(key=lambda x: x[1], reverse=True)

            for ticker, score, row in candidates[:available_slots]:
                remaining = MAX_POSITIONS - len(open_positions)
                if remaining <= 0 or cash <= 0:
                    break

                allocation = cash / remaining
                shares     = allocation / row["Close"]
                cash      -= allocation

                open_positions[ticker] = {
                    "entry_date":    current_date,
                    "entry_price":   row["Close"],
                    "highest_price": row["Close"],
                    "shares":        shares,
                    "invested":      allocation,
                }

        # ── DAILY EQUITY ──────────────────────
        pv = cash
        for ticker, pos in open_positions.items():
            df = all_data[ticker]
            if current_date in df.index:
                pv += pos["shares"] * df.loc[current_date]["Close"]

        equity_curve.append(pv)
        date_index.append(current_date)

    return pd.DataFrame(trade_log), equity_curve, date_index

# ─────────────────────────────────────────────
#  NIFTY BENCHMARK
# ─────────────────────────────────────────────
def fetch_nifty(start, end):
    try:
        n     = yf.download("^NSEI", start=start, end=end, progress=False)
        close = n["Close"].squeeze()
        return close / close.iloc[0] * INITIAL_CAPITAL
    except:
        return None

# ─────────────────────────────────────────────
#  PRINT SUMMARY
# ─────────────────────────────────────────────
def print_summary(trades, equity_curve, date_index):
    print("\n" + "="*60)
    print("  BACKTEST RESULTS  (FCF Filter + Multi-Model Valuation)")
    print("="*60)

    final  = equity_curve[-1]
    years  = (date_index[-1] - date_index[0]).days / 365.25
    cagr   = calc_cagr(INITIAL_CAPITAL, final, years)
    mdd    = calc_max_drawdown(equity_curve)
    sharpe = calc_sharpe(equity_curve)

    print(f"  Period         : {date_index[0].date()} → {date_index[-1].date()}")
    print(f"  Initial Capital: ₹{INITIAL_CAPITAL:,.0f}")
    print(f"  Final Capital  : ₹{final:,.0f}")
    print(f"  Total Return   : {(final/INITIAL_CAPITAL-1)*100:.1f}%")
    print(f"  CAGR           : {cagr*100:.1f}%")
    print(f"  Max Drawdown   : {mdd*100:.1f}%")
    print(f"  Sharpe Ratio   : {sharpe:.2f}")
    print(f"  Total Trades   : {len(trades)}")

    if not trades.empty:
        wins     = trades[trades["Profit"] > 0]
        losses   = trades[trades["Profit"] < 0]
        win_rate = len(wins) / len(trades)
        avg_win  = wins["Profit"].mean()   if len(wins)   > 0 else 0
        avg_loss = losses["Profit"].mean() if len(losses) > 0 else 0
        rr       = abs(avg_win / avg_loss) if avg_loss != 0 else float("inf")

        print(f"  Win Rate       : {win_rate*100:.1f}%")
        print(f"  Avg Win        : ₹{avg_win:,.0f}")
        print(f"  Avg Loss       : ₹{avg_loss:,.0f}")
        print(f"  Risk/Reward    : {rr:.2f}")
        print(f"  Avg Hold (days): {trades['Holding Days'].mean():.0f}")
        print(f"  Best Trade     : ₹{trades['Profit'].max():,.0f}")
        print(f"  Worst Trade    : ₹{trades['Profit'].min():,.0f}")

        print("\n  Exit Reason Breakdown:")
        for reason, cnt in trades["Exit Reason"].value_counts().items():
            avg_r = trades[trades["Exit Reason"] == reason]["Return %"].mean()
            print(f"    {reason:<25} {cnt:>4} trades  |  avg return: {avg_r:.1f}%")

        print("\n  Avg FCF Yield of traded stocks:")
        print(f"    {trades['FCF Yield %'].mean():.2f}%")

        print("\n  Avg Profit by Market Cap:")
        cap_bins   = [0, 5_000, 50_000, 500_000, np.inf]
        cap_labels = ["Small Cap", "Mid Cap", "Large Cap", "Mega Cap"]
        trades["Cap Group"] = pd.cut(
            trades["Market Cap Cr"], bins=cap_bins, labels=cap_labels
        )
        print(trades.groupby("Cap Group", observed=True)["Profit"].mean().to_string())

    print("="*60 + "\n")

# ─────────────────────────────────────────────
#  SAVE TRADE LOG
# ─────────────────────────────────────────────
def save_trades(trades):
    if trades.empty:
        print("No trades to save.")
        return

    trades = trades.sort_values("Exit Date").copy()
    trades["Cumulative Profit"] = trades["Profit"].cumsum()
    trades["Equity"]            = INITIAL_CAPITAL + trades["Cumulative Profit"]

    # Strip timezone (Excel fix)
    for col in trades.select_dtypes(include=["datetimetz"]).columns:
        trades[col] = trades[col].dt.tz_localize(None)

    try:
        trades.to_excel("trade_log_fcf.xlsx", index=False)
        print("Trade log saved: trade_log_fcf.xlsx")
    except ImportError:
        trades.to_csv("trade_log_fcf.csv", index=False)
        print("Trade log saved: trade_log_fcf.csv")

# ─────────────────────────────────────────────
#  PLOT
# ─────────────────────────────────────────────
def plot_results(equity_curve, date_index, trades, nifty=None):
    fig = plt.figure(figsize=(16, 13))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.35)

    # 1. Equity curve
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(date_index, equity_curve, color="#534AB7", lw=1.5, label="Strategy (FCF + Valuation)")
    if nifty is not None:
        na = nifty.reindex(pd.DatetimeIndex(date_index), method="ffill")
        ax1.plot(date_index, na.values, color="#1D9E75", lw=1.2,
                 linestyle="--", label="Nifty 50 (buy & hold)")
    ax1.set_title("Portfolio Equity Curve vs Nifty 50", fontsize=13)
    ax1.set_ylabel("Portfolio Value (₹)")
    ax1.legend(); ax1.grid(alpha=0.2)

    if not trades.empty:
        # 2. Drawdown
        ax2 = fig.add_subplot(gs[1, 0])
        eq   = np.array(equity_curve)
        peak = np.maximum.accumulate(eq)
        dd   = (eq - peak) / peak * 100
        ax2.fill_between(date_index, dd, 0, color="#E24B4A", alpha=0.4)
        ax2.set_title("Drawdown %", fontsize=11)
        ax2.set_ylabel("%"); ax2.grid(alpha=0.2)

        # 3. Monthly P&L
        ax3 = fig.add_subplot(gs[1, 1])
        trades["Exit Date"] = pd.to_datetime(trades["Exit Date"])
        monthly = trades.groupby(trades["Exit Date"].dt.to_period("M"))["Profit"].sum()
        colors  = ["#1D9E75" if v >= 0 else "#E24B4A" for v in monthly.values]
        ax3.bar(range(len(monthly)), monthly.values, color=colors)
        ax3.set_title("Monthly P&L", fontsize=11)
        ax3.set_ylabel("₹"); ax3.grid(alpha=0.2, axis="y")
        ax3.set_xticks([]); ax3.axhline(0, color="black", lw=0.8)

        # 4. FCF Yield distribution of traded stocks
        ax4 = fig.add_subplot(gs[2, 0])
        ax4.hist(trades["FCF Yield %"].dropna(), bins=25,
                 color="#1D9E75", alpha=0.7, edgecolor="white")
        ax4.axvline(trades["FCF Yield %"].mean(), color="#E24B4A", lw=1.5,
                    linestyle="--", label=f'Mean: {trades["FCF Yield %"].mean():.1f}%')
        ax4.set_title("FCF Yield % of Traded Stocks", fontsize=11)
        ax4.set_xlabel("FCF Yield %"); ax4.legend(); ax4.grid(alpha=0.2)

        # 5. Return distribution
        ax5 = fig.add_subplot(gs[2, 1])
        ax5.hist(trades["Return %"], bins=40,
                 color="#534AB7", alpha=0.7, edgecolor="white")
        ax5.axvline(0, color="#E24B4A", lw=1.5, linestyle="--")
        ax5.axvline(trades["Return %"].mean(), color="#1D9E75", lw=1.5,
                    linestyle="--", label=f'Mean: {trades["Return %"].mean():.1f}%')
        ax5.set_title("Trade Return Distribution", fontsize=11)
        ax5.set_xlabel("Return %"); ax5.legend(); ax5.grid(alpha=0.2)

    plt.suptitle("NSE Backtest — FCF Filter + Graham / DCF / EV·FCF Valuation",
                 fontsize=13, fontweight="bold", y=1.01)
    plt.savefig("backtest_fcf_results.png", dpi=150, bbox_inches="tight")
    print("Chart saved: backtest_fcf_results.png")
    plt.show()

# ─────────────────────────────────────────────
#  MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("\n" + "="*60)
    print("  NSE BACKTEST — FCF Filter + Multi-Model Valuation")
    print("="*60 + "\n")

    tickers  = load_nse_tickers()
    print(f"Total NSE tickers: {len(tickers)}")

    all_data = load_all_data(tickers)

    if not all_data:
        print("ERROR: No FCF-positive stocks found. Check EQUITY_L.csv.")
        exit(1)

    trades, equity_curve, date_index = run_backtest(all_data)

    print_summary(trades, equity_curve, date_index)

    nifty = fetch_nifty(date_index[0], date_index[-1])

    save_trades(trades)
    plot_results(equity_curve, date_index, trades, nifty)


  NSE BACKTEST — FCF Filter + Multi-Model Valuation

Total NSE tickers: 2224

Fetching 2224 tickers (20 workers) with FCF filter …

  [100/2224]  4%  |  passed FCF filter: 35  |  ETA: 1.5 min


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ARISINFRA.NS"}}}


  [200/2224]  9%  |  passed FCF filter: 71  |  ETA: 1.2 min
  [300/2224]  13%  |  passed FCF filter: 109  |  ETA: 1.2 min
  [400/2224]  18%  |  passed FCF filter: 147  |  ETA: 1.1 min
  [500/2224]  22%  |  passed FCF filter: 192  |  ETA: 1.0 min
  [600/2224]  27%  |  passed FCF filter: 230  |  ETA: 0.9 min


HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}


  [700/2224]  31%  |  passed FCF filter: 261  |  ETA: 0.7 min


...

  [2000/2224]  90%  |  passed FCF filter: 721  |  ETA: 0.1 min


HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"User is unable to access this feature - https://bit.ly/yahoo-finance-api-feedback"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"User is unable to access this feature - https://bit.ly/yahoo-finance-api-feedback"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"User is unable to access this feat

  [2100/2224]  94%  |  passed FCF filter: 757  |  ETA: 0.0 min


HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"error":{"code":"Unauthorized","description":"Invalid Crumb"}}}
HTTP Error 401: {"finance":{"result":null,"err

  [2200/2224]  99%  |  passed FCF filter: 778  |  ETA: 0.0 min
  [2224/2224]  100%  |  passed FCF filter: 788  |  ETA: 0.0 min

FCF-positive stocks loaded: 788 / 2224  (35.4%)  in 0.6 min

Running backtest over 2477 trading days …

